# 평가 회귀를 CI에 연결하기

Ch15와 Ch29에서 배운 서로 다른 평가 기준을 하나의 종료 코드로 연결합니다.

### 이 준비 셀이 하는 일

첫 셀의 `Path` 탐색은 어느 상위 폴더에서 열어도 Ch33의 `pyproject.toml`을 기준으로 실습 폴더를 찾습니다. 뒤의 `subprocess.run(..., cwd=lab, capture_output=True)`은 명령을 별도 프로세스로 실행하고 화면 문구와 종료 코드를 함께 반환합니다. 이 보조 코드는 CI를 흉내 내는 공통 실행 경계이며, 각 절의 판정은 다음 셀에서 해당 명령의 실제 반환값을 읽어 결정합니다.

In [1]:
from pathlib import Path
import json, subprocess, sys
lab = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "ch33-cicd" and (p / "pyproject.toml").is_file()), Path("labs/part7/ch33-cicd")).resolve()
baseline = json.loads((lab / "data/regression-baseline.json").read_text())
print(baseline)

{'source': ['Ch15', 'Ch29'], 'source_files': {'labs/part4/ch15-evaluation-advanced/data/evaluation-cases.json': 'dc48b1bc49d26d511ca2141b102cefbae1b395a06faaa8d21a61c2174ec3b2d1', 'labs/part6/ch29-agent-evaluation/notebooks/04-운영-지표와-Ch18-궤적-비교.ipynb': '71fa56efb956b5218732c75bcf030ccf401e2335217d6b23fbf6c2ff2322d93a'}, 'note': '원본의 평가 항목을 CI 연결 연습에 맞게 축약한 교육용 기준선이며 원본 측정값 발췌가 아닙니다.', 'groundedness': 0.9, 'trajectory_extra_calls': 1, 'cases': 4}


### 바로 다음 코드를 읽는 방법

평가 회귀 검사는 현재 결과를 저장된 기준선과 비교해 배포 가능 여부를 종료 코드로 바꿉니다. 이 예제는 근거성 점수와 추가 도구 호출 수를 함께 검사합니다. 임시 복사본에서 답과 도구 호출을 악화시키면 평가 스크립트가 0이 아닌 코드를 반환하고, 원래 구현에서는 0을 반환합니다.

In [2]:
result = subprocess.run([sys.executable, "scripts/evaluate_regression.py"], cwd=lab, text=True, capture_output=True)
print(result.stdout.strip())
print("exit_code=", result.returncode)
assert result.returncode == 0

groundedness=1.00 required=0.90
trajectory_extra_calls=0 allowed=1
groundedness: PASS
trajectory: PASS
exit_code= 0


## 실패 재현

임시 저장소에 Groundedness와 추가 호출 수가 모두 나빠진 현재 결과를 둡니다.

In [3]:
import tempfile, shutil
with tempfile.TemporaryDirectory() as temp:
    t = Path(temp)
    shutil.copytree(lab / "scripts", t / "scripts")
    shutil.copytree(lab / "data", t / "data")
    source = (lab / "cicd_demo.py").read_text()
    source = source.replace("7권까지 14일", "8권까지 10일")
    source = source.replace("return responses[qid], tools", "return responses[qid], tools + ['retrieve', 'retrieve']")
    (t / "cicd_demo.py").write_text(source)
    failed = subprocess.run([sys.executable, "scripts/evaluate_regression.py"], cwd=t, text=True, capture_output=True)
    print(failed.stdout.strip(), "exit_code=", failed.returncode)
    assert failed.returncode != 0

groundedness=0.75 required=0.90
trajectory_extra_calls=8 allowed=1
groundedness: FAIL
trajectory: FAIL exit_code= 1


## 수정

기준을 만족하는 결과로 바꾸면 두 검사가 모두 통과합니다.

### 이 단계에서 구분할 범위

기준선은 영원한 정답이 아니라 승인된 비교점입니다. 질문 집합이나 평가 방식이 바뀌면 점수 변화가 모델 회귀인지 평가 변경인지 분리해야 합니다. 평균 점수만 보면 특정 중요 질문의 큰 하락이 가려질 수 있으므로 핵심 사례별 하한과 전체 집계를 함께 둡니다.

In [4]:
with tempfile.TemporaryDirectory() as temp:
    t = Path(temp)
    shutil.copytree(lab / "scripts", t / "scripts")
    shutil.copytree(lab / "data", t / "data")
    shutil.copy(lab / "cicd_demo.py", t / "cicd_demo.py")
    passed = subprocess.run([sys.executable, "scripts/evaluate_regression.py"], cwd=t, text=True, capture_output=True)
    print(passed.stdout.strip(), "exit_code=", passed.returncode)
    assert passed.returncode == 0

groundedness=1.00 required=0.90
trajectory_extra_calls=0 allowed=1
groundedness: PASS
trajectory: PASS exit_code= 0
